# Clean Operator Data Reports
Code author: Audrey McManemin

Edited from code written by Sahar H. El Abbadi

Date started: 2024-11-14
Date last edited: 2024-11-14


### Notes
- All raw report inputs are saved in 00_raw_reports. No changes are manually made to operator reports (except for date propagation in release schedule). All cleaning is handled in Python.

In [1]:
# Imports
import pandas as pd
import numpy as np
import pathlib
from methods_source import RELEASE_NUMBER_LIST
from methods_clean_operator_reports import operator_qc, stanford_qc, strict_qc

# Load and clean raw data submitted by operators
### Notes on formatting:
- Operators added their own QC indicators, thus not all columns are uniform across reports
- Values left in the Excel file are replaced during import into PyCharm with "nan"
- Naming convention for dataframes: operator

## Notes on Cleaning Operator Data

### Generate data frame with the following columns:
- Operator: name of operator (Aeromon, SeekOps, GSMA)
- Week: week that the operator participated in (1, 2, 3, 4)
- ReleaseID: release number for that week, corresponding to the master schedule of all the releases (not just for which that operator measured)
- DateOfSurvey: date in YYYY-MM-DD format
- SurveyStartTime: start time of survey in local time (UTC+2)
- SurveyEndTime: end time of survey in local time (UTC+2)
- QuantifiedPlume: boolean input, 1 indicates operator submitted a valid quantification estimate for this overpass (excludes quantification estimates that are provided but fail operator QC standards)
- EstimatedEmissionRate: estimated emissions in kgh
- EstimatedEmissionRateUpper: upper bound of uncertainty on quantification estimate
- EstimatedEmissionRateLower: lower bound of uncertainty on quantification estimate
- UncertaintyType: type of uncertainty for upper and lower values reported above
- OperatorWindspeed: operator reported windspeed in m/s
- QCFlag: operator specific QC flag
- OperatorKept: operator submitted estimates for this result
- EstimateType: if applicable, the method used for this estimate



## GHGSat 
### Submission Details
- Submitted W1 July 18, 2024
- Submitted W2 July 25, 2024
- Submitted W3 and W4 October 7, 2024

Data cleaning
- Submitted estimates for all tasked releases (if could not retrieve an image due to cloud cover then NA as estimate)


In [16]:
# %% GHGSat data cleaning

def clean_ghgsat(results, schedule, week):
    operator = 'GHGSat'
    num_releases = range(1, RELEASE_NUMBER_LIST[week] + 1) # for loop index
    release_list = [] # generating all new rows
    release_estimate_index = 0 # index for release estimate 
    
    # fill na with 'None' for quantification status
    schedule.loc[:, 'Quantification Status'] = schedule['Quantification Status'].fillna('None')
    
    for release in num_releases:
        if schedule.loc[release-1, "Measurement Taken"] == 'YES':
            if schedule.loc[release-1, "Quantification Status"] == 'Completed':
                quantified = True

            else:
                quantified = False

            emission_rate = results.loc[release_estimate_index, 'EstimatedEmissionRate']
            if np.isnan(emission_rate):
                emission_rate = 0
                emission_upper = 0
                emission_lower = 0
            else:
                emission_upper = results.loc[release_estimate_index, 'EstimatedEmissionRateUpper']
                emission_lower = results.loc[release_estimate_index, 'EstimatedEmissionRateLower']
            uncertainty_type = results.loc[release_estimate_index, 'UncertaintyType']
            windspeed = results.loc[release_estimate_index, "WindSpeed"],
            start_time = results.loc[release_estimate_index, "TimeStamp"]
            end_time = results.loc[release_estimate_index, "TimeStamp"]
            
            release_estimate_index += 1 # increment index to get next release estimate
        # make everything else zeros
        else:
            quantified = False
            emission_rate = 0
            emission_upper = 0
            emission_lower = 0
            uncertainty_type = np.nan
            windspeed = np.nan 
            start_time = schedule.loc[release-1, "Start Time"] 
            end_time = schedule.loc[release-1, "End Time"]
        
        QCflag = schedule.loc[release-1, "Explanation"]
        
        ## QC analysis
        measurement_taken = schedule.loc[release-1, "Measurement Taken"].lower()
        quantification_status = schedule.loc[release-1, "Quantification Status"].lower()
        operator_keep = operator_qc(measurement_taken, quantification_status)
        stanford_keep = stanford_qc(release, schedule)
        strict_qc_keep = strict_qc(measurement_taken, quantification_status)
        
        new_row = {
            'Operator': operator, 
            'Week': week,
            'DateOfSurvey': schedule.loc[release-1, "Date"],
            'ReleaseID': release,  
            'SurveyStartTime': start_time,
            'SurveyEndTime': end_time,
            'QuantifiedPlume': quantified,
            'EstimatedEmissionRate': emission_rate,
            'EstimatedEmissionRateUpper': emission_upper,
            'EstimatedEmissionRateLower': emission_lower,
            'UncertaintyType': uncertainty_type,
            'OperatorWindspeed': windspeed,
            'QCFLag': QCflag,
            'OperatorKeep': operator_keep,
            'StanfordKeep': stanford_keep,
            'StrictQCKeep': strict_qc_keep,
        }
        
        release_list.append(new_row)

    clean_df = pd.DataFrame(release_list)
    
    return clean_df

In [17]:
df_list = []
for week in [1, 2, 3, 4]:
    ghgsat_results_path = pathlib.PurePath('00_raw_reports/GHGSat/', f'GHGSat_results_W{week}.xlsx')
    ghgsat_results = pd.read_excel(ghgsat_results_path, sheet_name='Reported Data', engine='openpyxl')
    ghgsat_schedule_path = pathlib.PurePath('00_raw_reports/GHGSat/', f'GHGSat_schedule_W{week}.xlsx')
    ghgsat_schedule = pd.read_excel(ghgsat_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

    df = clean_ghgsat(ghgsat_results, ghgsat_schedule, week)
    df_list.append(df)

ghgsat_clean = pd.concat(df_list)
ghgsat_clean.rename(columns={'ReleaseID': 'WeeklyReleaseID'}, inplace=True)
ghgsat_clean = ghgsat_clean.reset_index()
ghgsat_clean['ReleaseID'] = ghgsat_clean.index + 1

# save data
ghgsat_clean.to_csv(pathlib.PurePath('01_clean_reports', 'GHGSat_clean.csv'), index=False)

## FAAM

In [23]:
# %% FAAM data cleaning

def clean_faam(results, schedule, week):
    operator = 'FAAM'
    num_releases = range(1, RELEASE_NUMBER_LIST[week] + 1) # for loop index
    release_list = [] # generating all new rows
    release_estimate_index = 0 # index for release estimate 
    
    # fill na with 'None' for quantification status
    schedule.loc[:, 'Quantification Status'] = schedule['Quantification Status'].fillna('None')

    for release in num_releases:
        if schedule.loc[release-1, "Measurement Taken"] == 'YES':
            if schedule.loc[release-1, "Quantification Status"] == 'Completed':
                quantified = True
                emission_rate = results.loc[release_estimate_index, 'EstimatedEmissionRate']
                emission_upper = results.loc[release_estimate_index, 'EstimatedEmissionRateUpper']
                emission_lower = results.loc[release_estimate_index, 'EstimatedEmissionRateLower']
                uncertainty_type = results.loc[release_estimate_index, 'UncertaintyType']
                windspeed = results.loc[release_estimate_index, "WindSpeed"]

                release_estimate_index = release_estimate_index + 1
            else:
                quantified = False
                emission_rate = 0
                emission_upper = 0
                emission_lower = 0
                uncertainty_type = np.nan
                windspeed = np.nan 

        # make everything else zeros
        else:
            quantified = np.nan
            emission_rate = np.nan
            emission_upper = np.nan
            emission_lower = np.nan
            uncertainty_type = np.nan
            windspeed = np.nan 

        ## QC analysis
        measurement_taken = schedule.loc[release-1, "Measurement Taken"].lower()
        quantification_status = schedule.loc[release-1, "Quantification Status"].lower()
        operator_keep = operator_qc(measurement_taken, quantification_status)
        stanford_keep = stanford_qc(release, schedule)
        strict_qc_keep = strict_qc(measurement_taken, quantification_status)
        QCflag = schedule.loc[release-1, "Explanation"]
        
        start_time = schedule.loc[release-1, "Start Time"] 
        end_time = schedule.loc[release-1, "End Time"] 
        
        
        new_row = {
            'Operator': operator, 
            'Week': week,
            'DateOfSurvey': schedule.loc[release-1, "Date"],
            'ReleaseID': release,  
            'SurveyStartTime': start_time,
            'SurveyEndTime': end_time,
            'QuantifiedPlume': quantified,
            'EstimatedEmissionRate': emission_rate,
            'EstimatedEmissionRateUpper': emission_upper,
            'EstimatedEmissionRateLower': emission_lower,
            'UncertaintyType': uncertainty_type,
            'OperatorWindspeed': windspeed,
            'QCFLag': QCflag,
            'OperatorKeep': operator_keep,
            'StanfordKeep': stanford_keep,
            'StrictQCKeep': strict_qc_keep,
        }
        
        release_list.append(new_row)

    clean_df = pd.DataFrame(release_list)
    
    return clean_df

In [24]:
results = pd.read_excel('00_raw_reports/FAAM_results_submitted_10_16.xlsx', sheet_name='Reported Data', engine='openpyxl')
schedule = pd.read_excel('00_raw_reports/FAAM_release_schedule.xlsx', engine='openpyxl', skiprows=1, usecols='D:J')

df = clean_faam(results, schedule, week=3)

# save data
df.to_csv('01_clean_reports/faam_clean.csv', index=False)